# Notebook 5: The Evoformer

**Objective:** Understand the Evoformer -- the core neural network block of AlphaFold2 that iteratively refines MSA and pair representations through 48 stacked blocks.

---

The Evoformer is the computational heart of AlphaFold2. It takes the initial MSA representation $\mathbf{m} \in \mathbb{R}^{N_s \times N_r \times c_m}$ and pair representation $\mathbf{z} \in \mathbb{R}^{N_r \times N_r \times c_z}$ (constructed in Notebooks 3 and 4) and subjects them to 48 rounds of iterative refinement. Each round applies a carefully orchestrated sequence of attention, multiplicative updates, and feedforward transformations that allow the two representations to exchange information and converge toward a coherent picture of the protein's evolutionary constraints and three-dimensional structure.

This notebook dissects each operation inside a single Evoformer block, develops the mathematical formalism, and provides working implementations with synthetic data to build intuition for what each operation accomplishes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import matplotlib.patheffects as pe

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

---
## 1. The Evoformer at a Glance

AlphaFold2 stacks 48 identical Evoformer blocks. Each block updates two representations simultaneously:

- **MSA representation:** $\mathbf{m} \in \mathbb{R}^{N_s \times N_r \times 256}$, where $N_s$ is the number of sequences and $N_r$ the number of residues.
- **Pair representation:** $\mathbf{z} \in \mathbb{R}^{N_r \times N_r \times 128}$, encoding pairwise relationships between all residue pairs.

Within each block, the operations follow a specific order:

**MSA track:**
1. MSA Row-wise Gated Self-Attention with Pair Bias
2. MSA Column-wise Gated Self-Attention
3. MSA Transition (feedforward)

**MSA-to-Pair bridge:**
4. Outer Product Mean

**Pair track:**
5. Triangular Multiplicative Update (outgoing edges)
6. Triangular Multiplicative Update (incoming edges)
7. Triangular Self-Attention (starting node)
8. Triangular Self-Attention (ending node)
9. Pair Transition (feedforward)

Each operation uses residual connections and layer normalization. The pair representation also feeds back into the MSA track via the pair bias in step 1, completing a bidirectional communication loop.

In [ ]:
# ---------------------------------------------------------------
# Block diagram of one Evoformer block
# ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(14, 16))
ax.set_xlim(0, 14)
ax.set_ylim(0, 18)
ax.axis('off')
ax.set_aspect('equal')

# Color palette
c_msa = '#4C9BE8'       # MSA operations - blue
c_bridge = '#E8A84C'    # Bridge operation - orange
c_pair = '#6DC86D'      # Pair operations - green
c_transition = '#C87D6D' # Transitions - muted red
c_repr = '#D4D4D4'      # Representation boxes - light gray

def draw_box(ax, xy, w, h, label, color, fontsize=11):
    """Draw a rounded box with centered label."""
    box = FancyBboxPatch(xy, w, h, boxstyle='round,pad=0.15',
                         facecolor=color, edgecolor='#333333', linewidth=1.2, alpha=0.92)
    ax.add_patch(box)
    cx = xy[0] + w / 2
    cy = xy[1] + h / 2
    ax.text(cx, cy, label, ha='center', va='center', fontsize=fontsize,
            color='#1a1a1a', wrap=True)

def draw_arrow(ax, start, end, color='#555555', lw=1.5):
    """Draw an arrow between two points."""
    ax.annotate('', xy=end, xytext=start,
                arrowprops=dict(arrowstyle='->', color=color, lw=lw))

# Title
ax.text(7, 17.4, 'One Evoformer Block', ha='center', va='center',
        fontsize=16, color='#1a1a1a')

# Column headers
ax.text(3.5, 16.7, 'MSA Track', ha='center', va='center',
        fontsize=13, color=c_msa)
ax.text(10.5, 16.7, 'Pair Track', ha='center', va='center',
        fontsize=13, color='#3a8a3a')

# --- MSA representation input ---
draw_box(ax, (1.5, 15.5), 4, 0.7, r'MSA repr $\mathbf{m}$', c_repr, fontsize=12)

# --- Pair representation input ---
draw_box(ax, (8.5, 15.5), 4, 0.7, r'Pair repr $\mathbf{z}$', c_repr, fontsize=12)

# MSA operations
draw_arrow(ax, (3.5, 15.5), (3.5, 14.6))
draw_box(ax, (1.0, 13.7), 5, 0.7, 'Row Attention\n(with pair bias)', c_msa)

# Arrow from pair repr to row attention (pair bias)
draw_arrow(ax, (8.5, 15.85), (6.0, 14.1), color=c_bridge, lw=2.0)
ax.text(7.6, 15.1, 'pair bias', fontsize=9, color=c_bridge, fontstyle='italic', rotation=17)

draw_arrow(ax, (3.5, 13.7), (3.5, 12.8))
draw_box(ax, (1.0, 11.9), 5, 0.7, 'Column Attention', c_msa)

draw_arrow(ax, (3.5, 11.9), (3.5, 11.0))
draw_box(ax, (1.0, 10.1), 5, 0.7, 'MSA Transition', c_transition)

# Bridge: Outer product mean
draw_arrow(ax, (3.5, 10.1), (3.5, 9.2))
draw_box(ax, (1.0, 8.3), 5, 0.7, 'Outer Product Mean', c_bridge)

# Arrow from OPM to pair track
draw_arrow(ax, (6.0, 8.65), (8.5, 8.65), color=c_bridge, lw=2.0)

# Updated MSA output
draw_arrow(ax, (3.5, 8.3), (3.5, 7.4))
draw_box(ax, (1.5, 6.6), 4, 0.7, r'Updated $\mathbf{m}$', c_repr, fontsize=12)

# Pair track operations (positioned to receive OPM output)
draw_arrow(ax, (10.5, 15.5), (10.5, 14.6))

# "+" symbol for adding OPM to pair
ax.text(9.2, 8.65, '+', ha='center', va='center', fontsize=16, color=c_bridge)

draw_box(ax, (8.0, 7.5), 5, 0.7, 'Tri. Mult. Update (outgoing)', c_pair)
draw_arrow(ax, (10.5, 14.6), (10.5, 8.9))
ax.text(10.9, 11.5, '(pass through + OPM)', fontsize=9, color='#666666',
        rotation=90, ha='center', va='center')

draw_arrow(ax, (10.5, 7.5), (10.5, 6.6))
draw_box(ax, (8.0, 5.7), 5, 0.7, 'Tri. Mult. Update (incoming)', c_pair)

draw_arrow(ax, (10.5, 5.7), (10.5, 4.8))
draw_box(ax, (8.0, 3.9), 5, 0.7, 'Tri. Self-Attn (starting)', c_pair)

draw_arrow(ax, (10.5, 3.9), (10.5, 3.0))
draw_box(ax, (8.0, 2.1), 5, 0.7, 'Tri. Self-Attn (ending)', c_pair)

draw_arrow(ax, (10.5, 2.1), (10.5, 1.2))
draw_box(ax, (8.0, 0.3), 5, 0.7, 'Pair Transition', c_transition)

draw_arrow(ax, (10.5, 0.3), (10.5, -0.3))
draw_box(ax, (8.5, -1.1), 4, 0.7, r'Updated $\mathbf{z}$', c_repr, fontsize=12)

# Legend
legend_items = [
    (c_msa, 'MSA Attention'),
    (c_pair, 'Pair Operations (triangular)'),
    (c_bridge, 'MSA-Pair Bridge'),
    (c_transition, 'Transition (feedforward)'),
]
for idx, (color, label) in enumerate(legend_items):
    y_pos = -0.3 - idx * 0.55
    box = FancyBboxPatch((1.0, y_pos - 0.15), 0.8, 0.35,
                         boxstyle='round,pad=0.05', facecolor=color,
                         edgecolor='#333333', linewidth=0.8, alpha=0.9)
    ax.add_patch(box)
    ax.text(2.1, y_pos + 0.02, label, ha='left', va='center', fontsize=11, color='#333333')

ax.set_ylim(-2.6, 18)
plt.tight_layout()
plt.show()

The diagram above shows the complete data flow within a single Evoformer block. The left column processes the MSA representation and the right column processes the pair representation. The two critical cross-talk points are:

1. **Pair $\to$ MSA:** The pair bias injected into MSA row-wise attention (top-left arrow crossing from pair to MSA).
2. **MSA $\to$ Pair:** The Outer Product Mean that converts MSA covariation signals into pair updates (middle horizontal arrow).

This bidirectional communication, repeated 48 times, is what allows AlphaFold2 to jointly reason about evolutionary information and 3D structure.

---
## 2. MSA Row Attention with Pair Bias

The first operation in each Evoformer block is **row-wise gated self-attention** on the MSA representation, augmented with an additive bias derived from the pair representation. For each sequence $s$ in the MSA, attention is computed along the residue dimension (i.e., each sequence attends to all residue positions).

### Standard multi-head attention

For head $h$, the query, key, and value projections at sequence $s$ and position $i$ are:

$$
q_{si}^h = W_Q^h \, \text{LayerNorm}(\mathbf{m}_{si}), \quad
k_{si}^h = W_K^h \, \text{LayerNorm}(\mathbf{m}_{si}), \quad
v_{si}^h = W_V^h \, \text{LayerNorm}(\mathbf{m}_{si})
$$

### The pair bias

The attention logits for sequence $s$, positions $i$ and $j$, head $h$ are:

$$
\alpha_{sij}^h = \frac{1}{\sqrt{d_h}} \, (q_{si}^h)^T k_{sj}^h \;+\; b_{ij}^h
$$

where the **pair bias** is:

$$
b_{ij}^h = \text{Linear}^h(\text{LayerNorm}(\mathbf{z}_{ij})) \in \mathbb{R}
$$

Note that $b_{ij}^h$ depends only on positions $i,j$ and **not** on the sequence index $s$. This means all sequences share the same structural bias, which is exactly the right inductive bias: the 3D structure is the same for all homologous sequences.

The attention weights are then:

$$
w_{sij}^h = \text{softmax}_j\!\left(\alpha_{sij}^h\right)
$$

and the gated output is:

$$
\mathbf{m}_{si} \leftarrow \mathbf{m}_{si} + \text{Linear}\!\left(\text{concat}_h\!\left[g_{si}^h \odot \sum_j w_{sij}^h \, v_{sj}^h\right]\right)
$$

where $g_{si}^h = \sigma(\text{Linear}^h(\mathbf{m}_{si}))$ is a per-head gating vector.

### Why pair bias matters

Without pair bias, row attention can only discover relationships between positions based on sequence content. With pair bias, the model can tell the attention mechanism: "positions $i$ and $j$ are structurally close, so pay more attention to their relationship." This is the primary mechanism by which the pair representation (encoding structural information) influences the MSA representation (encoding evolutionary information).

In [ ]:
# ---------------------------------------------------------------
# Simplified MSA Row Attention with and without Pair Bias
# ---------------------------------------------------------------
N_seq, N_res, c_m, c_z = 8, 20, 32, 16
n_heads = 4
d_h = c_m // n_heads  # 8

# Synthetic data
msa_repr = np.random.randn(N_seq, N_res, c_m).astype(np.float32) * 0.5
pair_repr = np.random.randn(N_res, N_res, c_z).astype(np.float32) * 0.3

# Inject a structural signal: positions 5,6 are close to 14,15
# (simulate a beta-sheet contact)
for i in [5, 6]:
    for j in [14, 15]:
        pair_repr[i, j, :] += 2.0
        pair_repr[j, i, :] += 2.0

# Also inject nearby contacts along the diagonal
for i in range(N_res):
    for j in range(max(0, i-2), min(N_res, i+3)):
        pair_repr[i, j, :] += 1.5

# Random weight matrices
W_Q = np.random.randn(n_heads, c_m, d_h) * 0.1
W_K = np.random.randn(n_heads, c_m, d_h) * 0.1
W_bias = np.random.randn(n_heads, c_z) * 0.3  # projects pair repr to scalar per head

def softmax(x, axis=-1):
    e = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e / np.sum(e, axis=axis, keepdims=True)

def compute_row_attention(msa, pair, W_Q, W_K, W_bias, use_pair_bias=True):
    """Compute attention maps for all heads, averaged over sequences."""
    N_s, N_r, _ = msa.shape
    n_h = W_Q.shape[0]
    d = W_Q.shape[2]
    
    # Compute pair bias for each head: (n_heads, N_r, N_r)
    bias = np.zeros((n_h, N_r, N_r))
    if use_pair_bias:
        for h in range(n_h):
            bias[h] = pair @ W_bias[h]  # (N_r, N_r, c_z) @ (c_z,) -> (N_r, N_r)
    
    # Attention maps: average over sequences
    attn_maps = np.zeros((n_h, N_r, N_r))
    for h in range(n_h):
        for s in range(N_s):
            Q = msa[s] @ W_Q[h]  # (N_r, d)
            K = msa[s] @ W_K[h]  # (N_r, d)
            logits = (Q @ K.T) / np.sqrt(d) + bias[h]  # (N_r, N_r)
            attn_maps[h] += softmax(logits)
        attn_maps[h] /= N_s
    
    return attn_maps

attn_no_bias = compute_row_attention(msa_repr, pair_repr, W_Q, W_K, W_bias, use_pair_bias=False)
attn_with_bias = compute_row_attention(msa_repr, pair_repr, W_Q, W_K, W_bias, use_pair_bias=True)

# Visualization
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for h in range(n_heads):
    im0 = axes[0, h].imshow(attn_no_bias[h], cmap='Blues', vmin=0, vmax=0.2)
    axes[0, h].set_title(f'Head {h+1} (no bias)', fontsize=11)
    axes[0, h].set_xlabel('Key position', fontsize=11)
    if h == 0:
        axes[0, h].set_ylabel('Query position', fontsize=11)
    
    im1 = axes[1, h].imshow(attn_with_bias[h], cmap='Blues', vmin=0, vmax=0.2)
    axes[1, h].set_title(f'Head {h+1} (with pair bias)', fontsize=11)
    axes[1, h].set_xlabel('Key position', fontsize=11)
    if h == 0:
        axes[1, h].set_ylabel('Query position', fontsize=11)

# Mark the injected contacts
for h in range(n_heads):
    for i in [5, 6]:
        for j in [14, 15]:
            axes[1, h].plot(j, i, 's', color='red', markersize=4, markerfacecolor='none', markeredgewidth=1.2)
            axes[1, h].plot(i, j, 's', color='red', markersize=4, markerfacecolor='none', markeredgewidth=1.2)

fig.suptitle('MSA Row Attention: Without vs. With Pair Bias\n'
             '(Red squares mark injected structural contacts at positions 5-6 / 14-15)',
             fontsize=13, y=1.04)
plt.tight_layout()
plt.show()

# Quantitative comparison
print('Average attention at contact positions (5-6, 14-15):')
contact_attn_no = np.mean([attn_no_bias[:, i, j].mean() for i in [5,6] for j in [14,15]])
contact_attn_yes = np.mean([attn_with_bias[:, i, j].mean() for i in [5,6] for j in [14,15]])
print(f'  Without pair bias: {contact_attn_no:.4f}')
print(f'  With pair bias:    {contact_attn_yes:.4f}')
print(f'  Ratio (with/without): {contact_attn_yes / contact_attn_no:.2f}x')

The bottom row shows clear enhancement of attention at the structurally relevant positions (red squares). The pair bias acts as a structural prior, telling the attention mechanism where to look based on spatial proximity information accumulated in the pair representation. Without this bias (top row), attention is driven purely by sequence content and tends to be more diffuse.

---
## 3. MSA Column Attention

After row-wise attention processes each sequence along the residue dimension, **column-wise attention** processes each residue position along the sequence dimension. For a given residue position $i$, all $N_s$ sequences attend to each other:

$$
q_{si}^h = W_Q^h \, \text{LayerNorm}(\mathbf{m}_{si}), \quad
k_{ti}^h = W_K^h \, \text{LayerNorm}(\mathbf{m}_{ti})
$$

$$
\alpha_{st,i}^h = \frac{1}{\sqrt{d_h}} (q_{si}^h)^T k_{ti}^h
$$

Note: there is **no pair bias** here, because there is no meaningful pairwise structure between sequences (unlike the spatial structure between residue positions).

Column attention serves a crucial role: it allows sequences to communicate about each position. If sequence 3 has an informative mutation at position $i$, column attention lets all other sequences learn about it. This is the mechanism that extracts **coevolutionary signal** from the MSA -- if two positions tend to mutate together across sequences, column attention at those positions will capture correlated patterns.

In [ ]:
# ---------------------------------------------------------------
# MSA Column Attention Visualization
# ---------------------------------------------------------------
N_seq, N_res, c_m = 8, 20, 32
n_heads_col = 4
d_h_col = c_m // n_heads_col

# Create MSA with structure: sequences 0-3 are similar, 4-7 are similar but different from 0-3
msa_col = np.random.randn(N_seq, N_res, c_m).astype(np.float32) * 0.3
# Group A (seq 0-3): correlated pattern at position 5
msa_col[0:4, 5, :16] += 1.5
# Group B (seq 4-7): different pattern at position 5
msa_col[4:8, 5, 16:] += 1.5
# Position 10: all sequences similar (conserved)
msa_col[:, 10, :] = np.random.randn(c_m) * 0.1 + 0.5
# Position 15: each sequence unique (highly variable)
msa_col[:, 15, :] = np.random.randn(N_seq, c_m) * 1.5

W_Q_col = np.random.randn(n_heads_col, c_m, d_h_col) * 0.15
W_K_col = np.random.randn(n_heads_col, c_m, d_h_col) * 0.15

def compute_column_attention(msa, W_Q, W_K, position):
    """Compute column attention at a specific residue position."""
    N_s = msa.shape[0]
    n_h = W_Q.shape[0]
    d = W_Q.shape[2]
    
    col_data = msa[:, position, :]  # (N_s, c_m)
    attn_maps = np.zeros((n_h, N_s, N_s))
    
    for h in range(n_h):
        Q = col_data @ W_Q[h]  # (N_s, d)
        K = col_data @ W_K[h]  # (N_s, d)
        logits = (Q @ K.T) / np.sqrt(d)
        attn_maps[h] = softmax(logits)
    
    return attn_maps.mean(axis=0)  # average over heads

positions_to_show = [5, 10, 15]
labels = [
    'Position 5\n(two groups: seq 0-3 vs 4-7)',
    'Position 10\n(conserved across all sequences)',
    'Position 15\n(highly variable)'
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, (pos, label) in enumerate(zip(positions_to_show, labels)):
    attn = compute_column_attention(msa_col, W_Q_col, W_K_col, pos)
    im = axes[idx].imshow(attn, cmap='YlOrRd', vmin=0, vmax=0.3)
    axes[idx].set_title(label, fontsize=11)
    axes[idx].set_xlabel('Key (sequence index)', fontsize=11)
    axes[idx].set_ylabel('Query (sequence index)', fontsize=11)
    axes[idx].set_xticks(range(N_seq))
    axes[idx].set_yticks(range(N_seq))
    plt.colorbar(im, ax=axes[idx], shrink=0.8)

    # Draw group boundaries for position 5
    if pos == 5:
        axes[idx].axhline(y=3.5, color='white', linewidth=2, linestyle='--')
        axes[idx].axvline(x=3.5, color='white', linewidth=2, linestyle='--')

fig.suptitle('Column Attention at Three Residue Positions\n(averaged over heads)', fontsize=13, y=1.03)
plt.tight_layout()
plt.show()

The three attention patterns reveal qualitatively different behaviors:

- **Position 5 (two groups):** Attention is block-diagonal -- sequences 0--3 attend strongly to each other, and sequences 4--7 attend strongly to each other. The network detects the two evolutionary sub-groups.
- **Position 10 (conserved):** Attention is relatively uniform because all sequences carry nearly the same information. There is little to distinguish one sequence from another.
- **Position 15 (variable):** Attention is more dispersed and irregular, as each sequence carries unique information at this position.

Column attention is the primary mechanism for extracting **coevolutionary correlation** patterns from the MSA.

---
## 4. Outer Product Mean: MSA to Pair Communication

The **Outer Product Mean (OPM)** is the critical bridge that transfers information from the MSA representation to the pair representation. It captures how pairs of positions covary across sequences in the MSA.

### Mathematical formulation

Given the MSA representation $\mathbf{m} \in \mathbb{R}^{N_s \times N_r \times c_m}$, we first project each element through two learned linear maps:

$$
a_{si} = \text{Linear}_a(\text{LayerNorm}(\mathbf{m}_{si})) \in \mathbb{R}^{c_a}, \quad
b_{si} = \text{Linear}_b(\text{LayerNorm}(\mathbf{m}_{si})) \in \mathbb{R}^{c_b}
$$

Then for each pair $(i, j)$, we compute the outer product of $a_{si}$ and $b_{sj}$ and average over all sequences:

$$
\mathbf{o}_{ij} = \frac{1}{N_s} \sum_{s=1}^{N_s} a_{si} \otimes b_{sj} = \frac{1}{N_s} \sum_{s=1}^{N_s} a_{si} \, b_{sj}^T \in \mathbb{R}^{c_a \times c_b}
$$

This is then flattened and projected:

$$
\Delta \mathbf{z}_{ij} = \text{Linear}(\text{flatten}(\mathbf{o}_{ij})) \in \mathbb{R}^{c_z}
$$

### Intuition

The outer product $a_{si} \, b_{sj}^T$ captures the **joint representation** of positions $i$ and $j$ in sequence $s$. Averaging over sequences computes a kind of **covariance**: if positions $i$ and $j$ covary across sequences (e.g., both mutate together to maintain a structural contact), the mean outer product will capture this correlated signal. This is conceptually similar to direct coupling analysis (DCA) methods that extract contact information from MSAs, but here it is a differentiable operation learned end-to-end.

In [ ]:
# ---------------------------------------------------------------
# Outer Product Mean: Step-by-Step Implementation and Visualization
# ---------------------------------------------------------------
N_seq, N_res, c_m = 12, 16, 32
c_a, c_b = 8, 8  # projection dimensions

# Create synthetic MSA with coevolutionary signal
msa_opm = np.random.randn(N_seq, N_res, c_m).astype(np.float32) * 0.3

# Inject coevolution: positions 3 and 11 covary (compensatory mutations)
for s in range(N_seq):
    signal = np.random.randn() * 1.5  # shared signal
    msa_opm[s, 3, :8] += signal
    msa_opm[s, 11, :8] += signal * 0.9 + np.random.randn() * 0.2

# Positions 6 and 7 covary (neighbors)
for s in range(N_seq):
    signal = np.random.randn() * 1.2
    msa_opm[s, 6, 8:16] += signal
    msa_opm[s, 7, 8:16] += signal * 0.95

# Projection weights
W_a = np.random.randn(c_m, c_a) * 0.2
W_b = np.random.randn(c_m, c_b) * 0.2

# Step 1: Project
a_proj = msa_opm @ W_a  # (N_seq, N_res, c_a)
b_proj = msa_opm @ W_b  # (N_seq, N_res, c_b)

# Step 2: Outer product for each sequence, for positions i=3, j=11
i_pos, j_pos = 3, 11
outer_products = np.zeros((N_seq, c_a, c_b))
for s in range(N_seq):
    outer_products[s] = np.outer(a_proj[s, i_pos], b_proj[s, j_pos])

# Step 3: Mean outer product
mean_op = outer_products.mean(axis=0)

# Step 4: Full OPM for all pairs
opm_full = np.zeros((N_res, N_res, c_a * c_b))
for i in range(N_res):
    for j in range(N_res):
        op = np.zeros((c_a, c_b))
        for s in range(N_seq):
            op += np.outer(a_proj[s, i], b_proj[s, j])
        op /= N_seq
        opm_full[i, j] = op.flatten()

# Frobenius norm of OPM as a summary
opm_strength = np.linalg.norm(opm_full, axis=-1)

# --- Visualization ---
fig = plt.figure(figsize=(16, 10))

# (1) MSA matrix (first 8 channels, averaged)
ax1 = fig.add_subplot(2, 3, 1)
msa_display = msa_opm[:, :, :8].mean(axis=-1)
im1 = ax1.imshow(msa_display, cmap='RdBu_r', aspect='auto')
ax1.set_title('(1) MSA representation\n(first 8 channels, averaged)', fontsize=11)
ax1.set_xlabel('Residue position', fontsize=11)
ax1.set_ylabel('Sequence index', fontsize=11)
plt.colorbar(im1, ax=ax1, shrink=0.7)

# (2) Projected vectors for positions 3 and 11
ax2 = fig.add_subplot(2, 3, 2)
x = np.arange(N_seq)
width = 0.35
ax2.bar(x - width/2, a_proj[:, i_pos, 0], width, label=f'$a_{{s,{i_pos}}}$ (dim 0)', color='#4C9BE8', alpha=0.8)
ax2.bar(x + width/2, b_proj[:, j_pos, 0], width, label=f'$b_{{s,{j_pos}}}$ (dim 0)', color='#E8A84C', alpha=0.8)
ax2.set_title(f'(2) Projected vectors\npos {i_pos} and pos {j_pos} (dim 0)', fontsize=11)
ax2.set_xlabel('Sequence index', fontsize=11)
ax2.set_ylabel('Projected value', fontsize=11)
ax2.legend(fontsize=10)

# (3) Outer products for first 4 sequences
ax3_positions = [(2, 3, 3), (2, 3, 6)]
for idx, (r, c, pos) in enumerate(ax3_positions):
    ax = fig.add_subplot(r, c, pos)
    if idx == 0:
        # Show outer products for 4 sequences stacked
        display_ops = np.concatenate([outer_products[s] for s in range(4)], axis=1)
        im = ax.imshow(display_ops, cmap='RdBu_r', aspect='auto')
        ax.set_title(f'(3) Outer products $a_{{s,{i_pos}}} \\otimes b_{{s,{j_pos}}}$\n(seq 0-3, side by side)', fontsize=11)
        ax.set_xlabel('Concatenated $c_b$ dimensions', fontsize=11)
        ax.set_ylabel('$c_a$ dimension', fontsize=11)
        for boundary in range(1, 4):
            ax.axvline(x=boundary * c_b - 0.5, color='white', linewidth=2)
        plt.colorbar(im, ax=ax, shrink=0.7)
    else:
        # Mean outer product
        im = ax.imshow(mean_op, cmap='RdBu_r', aspect='equal')
        ax.set_title(f'(3b) Mean outer product\n$\\mathbf{{o}}_{{{i_pos},{j_pos}}}$', fontsize=11)
        ax.set_xlabel('$c_b$ dimension', fontsize=11)
        ax.set_ylabel('$c_a$ dimension', fontsize=11)
        plt.colorbar(im, ax=ax, shrink=0.7)

# (4) Full OPM strength as pair update heatmap
ax4 = fig.add_subplot(2, 3, (4, 5))
im4 = ax4.imshow(opm_strength, cmap='hot_r')
ax4.set_title('(4) OPM pair update strength $||\\mathbf{o}_{ij}||_F$\n'
              '(covarying positions show high values)', fontsize=11)
ax4.set_xlabel('Residue $j$', fontsize=11)
ax4.set_ylabel('Residue $i$', fontsize=11)
plt.colorbar(im4, ax=ax4, shrink=0.7)

# Mark injected contacts
for (pi, pj) in [(3, 11), (11, 3), (6, 7), (7, 6)]:
    ax4.plot(pj, pi, 'o', color='cyan', markersize=8, markerfacecolor='none', markeredgewidth=2)

ax4.text(N_res + 1.8, 3, 'Injected\ncoevolution', fontsize=10, color='cyan', va='center')

plt.tight_layout()
plt.show()

The visualization traces the OPM computation step by step:

1. **Panel (1):** The raw MSA representation, where the injected coevolutionary signals at positions 3/11 and 6/7 are visible as correlated patterns across sequences.
2. **Panel (2):** The projected vectors $a_{s,3}$ and $b_{s,11}$ show correlated variation -- when one goes up, the other tends to follow.
3. **Panel (3):** Individual outer products for each sequence capture this correlation; the mean outer product averages out noise and retains the systematic signal.
4. **Panel (4):** The full OPM strength map clearly highlights the covarying position pairs (cyan circles), demonstrating that the outer product mean successfully transfers coevolutionary information from the MSA to the pair representation.

---
## 5. Triangular Multiplicative Updates

The pair representation $\mathbf{z}_{ij}$ can be interpreted as the edge between nodes $i$ and $j$ in a complete graph over residues. The **triangular multiplicative updates** enforce a form of the triangle inequality: information about the edge $(i,j)$ should be consistent with the information along edges $(i,k)$ and $(k,j)$ for all intermediate nodes $k$.

### Outgoing edges

The "outgoing" update aggregates information from edges that share a starting node with edges $(i,k)$ and $(j,k)$:

$$
\tilde{\mathbf{z}}_{ij} = \text{LayerNorm}(\mathbf{z}_{ij})
$$

$$
a_{ik} = \sigma\!\left(\text{Linear}_g^a(\tilde{\mathbf{z}}_{ik})\right) \odot \text{Linear}_v^a(\tilde{\mathbf{z}}_{ik})
$$

$$
b_{jk} = \sigma\!\left(\text{Linear}_g^b(\tilde{\mathbf{z}}_{jk})\right) \odot \text{Linear}_v^b(\tilde{\mathbf{z}}_{jk})
$$

$$
\mathbf{z}_{ij} \leftarrow \mathbf{z}_{ij} + \text{Linear}\!\left(\text{LayerNorm}\!\left(\sum_k a_{ik} \odot b_{jk}\right)\right) \odot g_{ij}
$$

where $g_{ij} = \sigma(\text{Linear}_g(\tilde{\mathbf{z}}_{ij}))$ is an output gate.

### Incoming edges

The "incoming" update is analogous but aggregates over edges that share an ending node, summing over edges $(k,i)$ and $(k,j)$:

$$
a_{ki} = \sigma\!\left(\text{Linear}_g^a(\tilde{\mathbf{z}}_{ki})\right) \odot \text{Linear}_v^a(\tilde{\mathbf{z}}_{ki})
$$

$$
b_{kj} = \sigma\!\left(\text{Linear}_g^b(\tilde{\mathbf{z}}_{kj})\right) \odot \text{Linear}_v^b(\tilde{\mathbf{z}}_{kj})
$$

$$
\mathbf{z}_{ij} \leftarrow \mathbf{z}_{ij} + \text{Linear}\!\left(\text{LayerNorm}\!\left(\sum_k a_{ki} \odot b_{kj}\right)\right) \odot g_{ij}
$$

### Triangle interpretation

For the **outgoing** update, consider the triangle formed by nodes $i$, $j$, $k$. The edges $(i \to k)$ and $(j \to k)$ both "go out" from $i$ and $j$ toward $k$. If both edges indicate proximity to $k$, this is evidence that $i$ and $j$ are also close -- so the edge $(i,j)$ should be updated.

For the **incoming** update, edges $(k \to i)$ and $(k \to j)$ both "come in" from $k$ to $i$ and $j$. If some node $k$ is close to both $i$ and $j$, this again constrains the $(i,j)$ edge.

In [ ]:
# ---------------------------------------------------------------
# Triangular Multiplicative Update (Outgoing) Implementation
# ---------------------------------------------------------------
N_res_tri = 12
c_z_tri = 16

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -20, 20)))

# Create pair representation with sparse contacts
pair_tri = np.random.randn(N_res_tri, N_res_tri, c_z_tri).astype(np.float32) * 0.2

# Inject known contacts: 2-5, 5-9, and we expect 2-9 to be reinforced
pair_tri[2, 5, :] += 2.0; pair_tri[5, 2, :] += 2.0
pair_tri[5, 9, :] += 2.0; pair_tri[9, 5, :] += 2.0
# Also: 1-3, 3-7 -> should reinforce 1-7
pair_tri[1, 3, :] += 2.0; pair_tri[3, 1, :] += 2.0
pair_tri[3, 7, :] += 2.0; pair_tri[7, 3, :] += 2.0

# Random projection weights
W_ga = np.random.randn(c_z_tri, c_z_tri) * 0.2
W_va = np.random.randn(c_z_tri, c_z_tri) * 0.2
W_gb = np.random.randn(c_z_tri, c_z_tri) * 0.2
W_vb = np.random.randn(c_z_tri, c_z_tri) * 0.2
W_out = np.random.randn(c_z_tri, c_z_tri) * 0.1
W_gate = np.random.randn(c_z_tri, c_z_tri) * 0.2

def tri_mult_outgoing(z, W_ga, W_va, W_gb, W_vb, W_out, W_gate):
    """Simplified triangular multiplicative update (outgoing)."""
    N = z.shape[0]
    
    # Compute gated projections
    a = sigmoid(z @ W_ga) * (z @ W_va)  # (N, N, c)
    b = sigmoid(z @ W_gb) * (z @ W_vb)  # (N, N, c)
    gate = sigmoid(z @ W_gate)            # (N, N, c)
    
    # Triangular update: z_ij += sum_k a_ik * b_jk
    # a_ik: shape (N, N, c) -- index as a[i, k]
    # b_jk: shape (N, N, c) -- index as b[j, k]
    update = np.zeros_like(z)
    for i in range(N):
        for j in range(N):
            # sum over k: a[i,k,:] * b[j,k,:]
            update[i, j] = np.sum(a[i] * b[j], axis=0)  # element-wise mult then sum over k
    
    # Project and gate
    update_proj = update @ W_out
    return z + update_proj * gate

pair_before = pair_tri.copy()
pair_after = tri_mult_outgoing(pair_tri, W_ga, W_va, W_gb, W_vb, W_out, W_gate)

# Summary: norm of pair representation
norm_before = np.linalg.norm(pair_before, axis=-1)
norm_after = np.linalg.norm(pair_after, axis=-1)
norm_change = norm_after - norm_before

# --- Visualization ---
fig = plt.figure(figsize=(16, 6))

# (1) Pair matrix before
ax1 = fig.add_subplot(1, 3, 1)
im1 = ax1.imshow(norm_before, cmap='YlOrRd', vmin=0)
ax1.set_title('(1) Pair repr before update\n$||\\mathbf{z}_{ij}||$', fontsize=11)
ax1.set_xlabel('Residue $j$', fontsize=11)
ax1.set_ylabel('Residue $i$', fontsize=11)
plt.colorbar(im1, ax=ax1, shrink=0.7)

# (2) Triangle diagram
ax2 = fig.add_subplot(1, 3, 2)
ax2.set_xlim(-0.5, 3.5)
ax2.set_ylim(-0.5, 3.5)
ax2.set_aspect('equal')
ax2.axis('off')
ax2.set_title('(2) Triangle interpretation\n(outgoing edges from $i$ and $j$)', fontsize=11)

# Draw triangle: i at bottom-left, j at bottom-right, k at top
nodes = {'i': (0.5, 0.5), 'j': (2.5, 0.5), 'k': (1.5, 2.8)}
# Edges
# i -> k (outgoing from i)
ax2.annotate('', xy=nodes['k'], xytext=nodes['i'],
             arrowprops=dict(arrowstyle='->', color='#4C9BE8', lw=2.5))
ax2.text(0.6, 1.8, '$a_{ik}$', fontsize=13, color='#4C9BE8')

# j -> k (outgoing from j)
ax2.annotate('', xy=nodes['k'], xytext=nodes['j'],
             arrowprops=dict(arrowstyle='->', color='#E8A84C', lw=2.5))
ax2.text(2.2, 1.8, '$b_{jk}$', fontsize=13, color='#E8A84C')

# i -- j (updated edge)
ax2.annotate('', xy=nodes['j'], xytext=nodes['i'],
             arrowprops=dict(arrowstyle='<->', color='#CC4444', lw=3))
ax2.text(1.5, 0.15, '$\\mathbf{z}_{ij}$ (updated)', fontsize=12, color='#CC4444', ha='center')

# Node labels
for name, (x, y) in nodes.items():
    ax2.plot(x, y, 'o', markersize=20, color='#333333', zorder=5)
    ax2.text(x, y, f'${name}$', fontsize=14, color='white', ha='center', va='center', zorder=6)

ax2.text(1.5, 3.2, '$\\sum_k a_{ik} \\odot b_{jk}$', fontsize=13, ha='center', color='#333333')

# (3) Pair matrix after
ax3 = fig.add_subplot(1, 3, 3)
im3 = ax3.imshow(norm_change, cmap='RdBu_r')
ax3.set_title('(3) Change in pair repr\n$||\\mathbf{z}^{\\prime}_{ij}|| - ||\\mathbf{z}_{ij}||$', fontsize=11)
ax3.set_xlabel('Residue $j$', fontsize=11)
ax3.set_ylabel('Residue $i$', fontsize=11)
plt.colorbar(im3, ax=ax3, shrink=0.7)

# Mark expected transitive contacts
for (pi, pj) in [(2, 9), (9, 2), (1, 7), (7, 1)]:
    ax3.plot(pj, pi, 's', color='lime', markersize=10, markerfacecolor='none', markeredgewidth=2)

fig.text(0.85, 0.02, 'Green squares: expected\ntransitive contacts', fontsize=10, color='green',
         ha='center', va='bottom')

plt.tight_layout()
plt.show()

# Print transitive contact strengths
print('Transitive contact analysis:')
print(f'  Direct contacts: (2,5)={norm_before[2,5]:.2f}, (5,9)={norm_before[5,9]:.2f}')
print(f'  Transitive (2,9) before: {norm_before[2,9]:.2f}, after: {norm_after[2,9]:.2f}, change: {norm_change[2,9]:+.2f}')
print(f'  Direct contacts: (1,3)={norm_before[1,3]:.2f}, (3,7)={norm_before[3,7]:.2f}')
print(f'  Transitive (1,7) before: {norm_before[1,7]:.2f}, after: {norm_after[1,7]:.2f}, change: {norm_change[1,7]:+.2f}')

The triangular multiplicative update propagates contact information through intermediate nodes. In the example above:

- Residues 2 and 5 are in contact, and residues 5 and 9 are in contact. Through the triangle $(2, 5, 9)$, the update strengthens the evidence for a $(2, 9)$ contact.
- Similarly, contacts $(1, 3)$ and $(3, 7)$ propagate to suggest a $(1, 7)$ contact.

The outgoing and incoming variants provide complementary views of triangle consistency, together ensuring that the pair representation converges to a globally consistent set of pairwise relationships.

---
## 6. Triangular Self-Attention

While triangular multiplicative updates enforce consistency through element-wise gating and summation, **triangular self-attention** uses the attention mechanism to achieve a similar goal with greater expressiveness.

### Starting-node attention

For a fixed starting node $i$, we perform self-attention over the ending nodes $j$. The key innovation is that the attention bias comes from the pair representation itself:

$$
\alpha_{ij,ik}^h = \frac{1}{\sqrt{d_h}} (q_{ij}^h)^T k_{ik}^h + b_{jk}^h
$$

where $b_{jk}^h = \text{Linear}^h(\mathbf{z}_{jk})$. This means when updating the edge $(i,j)$, the model can attend to edge $(i,k)$ with a bias from edge $(j,k)$ -- again forming a triangle.

### Ending-node attention

For a fixed ending node $j$, attention is over starting nodes $i$:

$$
\alpha_{ij,kj}^h = \frac{1}{\sqrt{d_h}} (q_{ij}^h)^T k_{kj}^h + b_{ik}^h
$$

where $b_{ik}^h = \text{Linear}^h(\mathbf{z}_{ik})$.

### Interpretation

- **Starting-node attention** (row-wise on the pair matrix): for row $i$, the elements $\mathbf{z}_{ij}$ and $\mathbf{z}_{ik}$ attend to each other, biased by $\mathbf{z}_{jk}$. This captures: "if $i$ is close to both $j$ and $k$, then the relationship between $j$ and $k$ should influence how we update $\mathbf{z}_{ij}$."

- **Ending-node attention** (column-wise on the pair matrix): for column $j$, the elements $\mathbf{z}_{ij}$ and $\mathbf{z}_{kj}$ attend to each other, biased by $\mathbf{z}_{ik}$.

In [ ]:
# ---------------------------------------------------------------
# Triangular Self-Attention: Starting vs Ending Node Patterns
# ---------------------------------------------------------------
N_res_ta = 12
c_z_ta = 16
n_heads_ta = 4
d_h_ta = c_z_ta // n_heads_ta

# Pair representation with structure
pair_ta = np.random.randn(N_res_ta, N_res_ta, c_z_ta).astype(np.float32) * 0.3
# Add contacts: helix (i, i+4) pattern
for i in range(N_res_ta - 4):
    pair_ta[i, i+4, :] += 1.5
    pair_ta[i+4, i, :] += 1.5

W_Q_ta = np.random.randn(n_heads_ta, c_z_ta, d_h_ta) * 0.2
W_K_ta = np.random.randn(n_heads_ta, c_z_ta, d_h_ta) * 0.2
W_bias_ta = np.random.randn(n_heads_ta, c_z_ta) * 0.3

def tri_self_attention_starting(z, row_i, W_Q, W_K, W_bias):
    """Starting-node triangular self-attention for a fixed row i."""
    N = z.shape[0]
    n_h = W_Q.shape[0]
    d = W_Q.shape[2]
    
    row_data = z[row_i]  # (N, c_z) -- all edges starting from i
    
    attn_maps = np.zeros((n_h, N, N))
    for h in range(n_h):
        Q = row_data @ W_Q[h]  # (N, d)
        K = row_data @ W_K[h]  # (N, d)
        logits = (Q @ K.T) / np.sqrt(d)  # (N, N)
        # Triangular bias: b_jk from z[j,k]
        bias = z @ W_bias[h]  # (N, N) -- bias[j,k]
        logits += bias
        attn_maps[h] = softmax(logits)
    
    return attn_maps.mean(axis=0)

def tri_self_attention_ending(z, col_j, W_Q, W_K, W_bias):
    """Ending-node triangular self-attention for a fixed column j."""
    N = z.shape[0]
    n_h = W_Q.shape[0]
    d = W_Q.shape[2]
    
    col_data = z[:, col_j]  # (N, c_z) -- all edges ending at j
    
    attn_maps = np.zeros((n_h, N, N))
    for h in range(n_h):
        Q = col_data @ W_Q[h]  # (N, d)
        K = col_data @ W_K[h]  # (N, d)
        logits = (Q @ K.T) / np.sqrt(d)  # (N, N)
        # Triangular bias: b_ik from z[i,k]
        bias = z @ W_bias[h]  # (N, N) -- bias[i,k]
        logits += bias
        attn_maps[h] = softmax(logits)
    
    return attn_maps.mean(axis=0)

# Compute for specific rows/columns
test_nodes = [2, 5, 8]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for idx, node in enumerate(test_nodes):
    # Starting-node attention (row-wise)
    attn_start = tri_self_attention_starting(pair_ta, node, W_Q_ta, W_K_ta, W_bias_ta)
    im0 = axes[0, idx].imshow(attn_start, cmap='Purples', vmin=0, vmax=0.25)
    axes[0, idx].set_title(f'Starting node $i={node}$\nAttention among columns', fontsize=11)
    axes[0, idx].set_xlabel('Key (ending node $k$)', fontsize=11)
    axes[0, idx].set_ylabel('Query (ending node $j$)', fontsize=11)
    plt.colorbar(im0, ax=axes[0, idx], shrink=0.7)
    
    # Ending-node attention (column-wise)
    attn_end = tri_self_attention_ending(pair_ta, node, W_Q_ta, W_K_ta, W_bias_ta)
    im1 = axes[1, idx].imshow(attn_end, cmap='Greens', vmin=0, vmax=0.25)
    axes[1, idx].set_title(f'Ending node $j={node}$\nAttention among rows', fontsize=11)
    axes[1, idx].set_xlabel('Key (starting node $k$)', fontsize=11)
    axes[1, idx].set_ylabel('Query (starting node $i$)', fontsize=11)
    plt.colorbar(im1, ax=axes[1, idx], shrink=0.7)

fig.suptitle('Triangular Self-Attention: Starting Node (top, purple) vs. Ending Node (bottom, green)\n'
             '(averaged over heads; pair representation has helix-like i,i+4 contacts)',
             fontsize=13, y=1.03)
plt.tight_layout()
plt.show()

The two rows show fundamentally different attention patterns:

- **Starting-node attention (top, purple):** For a fixed starting node $i$, the attention operates over columns of the pair matrix. The triangular bias from $\mathbf{z}_{jk}$ means that ending nodes $j$ and $k$ that are themselves related will influence each other's updates.

- **Ending-node attention (bottom, green):** For a fixed ending node $j$, the attention operates over rows. The bias from $\mathbf{z}_{ik}$ means starting nodes that share connections to the same intermediate nodes will attend to each other.

Together, the starting-node and ending-node variants provide complementary views of triangular consistency. The attention mechanism is more expressive than the multiplicative update because it can learn complex, context-dependent weighting of the triangular constraints.

---
## 7. Transition Layers

Between the attention and multiplicative update operations, each Evoformer block applies **transition layers** (also called feedforward layers). These are simple two-layer networks applied independently to each position or pair:

$$
\text{Transition}(\mathbf{x}) = W_2 \, \text{ReLU}(W_1 \, \text{LayerNorm}(\mathbf{x}) + b_1) + b_2
$$

with an expansion factor of 4:

$$
W_1 \in \mathbb{R}^{4c \times c}, \quad W_2 \in \mathbb{R}^{c \times 4c}
$$

The residual connection is:

$$
\mathbf{x} \leftarrow \mathbf{x} + \text{Transition}(\mathbf{x})
$$

This is identical to the feedforward sub-layer in the original Transformer. The expansion to $4c$ dimensions allows the network to compute more complex nonlinear functions of the representation before projecting back down. These layers are applied:

1. After MSA column attention (MSA transition)
2. After triangular self-attention ending node (pair transition)

In [ ]:
# ---------------------------------------------------------------
# Transition Layer: Dimension Expansion/Contraction Diagram
# ---------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# (1) Schematic diagram of the transition layer
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.axis('off')
ax.set_title('Transition Layer Architecture', fontsize=13)

# Input
c_in = 128  # example channel dim
c_expand = 4 * c_in

# Draw rectangles representing dimensions
# Input: narrow
rect_in = FancyBboxPatch((1, 3), 0.8, 2, boxstyle='round,pad=0.05',
                          facecolor='#4C9BE8', edgecolor='#333', linewidth=1.2, alpha=0.85)
ax.add_patch(rect_in)
ax.text(1.4, 2.5, f'$c = {c_in}$', ha='center', fontsize=11)
ax.text(1.4, 5.3, 'Input', ha='center', fontsize=11)

# Expanded: wide
rect_exp = FancyBboxPatch((3.5, 1.5), 0.8, 5, boxstyle='round,pad=0.05',
                           facecolor='#E8A84C', edgecolor='#333', linewidth=1.2, alpha=0.85)
ax.add_patch(rect_exp)
ax.text(3.9, 0.8, f'$4c = {c_expand}$', ha='center', fontsize=11)
ax.text(3.9, 7.0, '$W_1 \\mathbf{x} + b_1$\n+ ReLU', ha='center', fontsize=11)

# Output: narrow again
rect_out = FancyBboxPatch((6, 3), 0.8, 2, boxstyle='round,pad=0.05',
                           facecolor='#6DC86D', edgecolor='#333', linewidth=1.2, alpha=0.85)
ax.add_patch(rect_out)
ax.text(6.4, 2.5, f'$c = {c_in}$', ha='center', fontsize=11)
ax.text(6.4, 5.3, 'Output', ha='center', fontsize=11)

# Arrows
# Input -> Expanded (fan out)
for y_start, y_end in [(3.5, 2.0), (4.0, 4.0), (4.5, 6.0)]:
    ax.annotate('', xy=(3.5, y_end), xytext=(1.8, y_start),
                arrowprops=dict(arrowstyle='->', color='#666', lw=1))

# Expanded -> Output (fan in)
for y_start, y_end in [(2.0, 3.5), (4.0, 4.0), (6.0, 4.5)]:
    ax.annotate('', xy=(6.0, y_end), xytext=(4.3, y_start),
                arrowprops=dict(arrowstyle='->', color='#666', lw=1))

# Residual arrow
ax.annotate('', xy=(7.5, 4.0), xytext=(1.4, 7.5),
            arrowprops=dict(arrowstyle='->', color='#CC4444', lw=2,
                          connectionstyle='arc3,rad=0.3'))
ax.text(5.0, 7.7, '+ residual', fontsize=11, color='#CC4444', ha='center')

rect_final = FancyBboxPatch((7.2, 3), 0.8, 2, boxstyle='round,pad=0.05',
                             facecolor='#9B6DC8', edgecolor='#333', linewidth=1.2, alpha=0.85)
ax.add_patch(rect_final)
ax.text(7.6, 2.5, f'$c = {c_in}$', ha='center', fontsize=11)
ax.text(7.6, 5.3, 'Final', ha='center', fontsize=11)

# (2) Actual activation distribution through transition layer
ax2 = axes[1]

c_demo = 32
c_exp_demo = 4 * c_demo
x_in = np.random.randn(200, c_demo) * 0.5

W1 = np.random.randn(c_demo, c_exp_demo) * np.sqrt(2.0 / c_demo)
b1 = np.zeros(c_exp_demo)
W2 = np.random.randn(c_exp_demo, c_demo) * np.sqrt(2.0 / c_exp_demo)
b2 = np.zeros(c_demo)

h_expanded = np.maximum(0, x_in @ W1 + b1)  # ReLU
x_out = h_expanded @ W2 + b2
x_final = x_in + x_out  # residual

# Plot distributions
ax2.hist(x_in.flatten(), bins=60, alpha=0.5, density=True, label='Input', color='#4C9BE8')
ax2.hist(h_expanded.flatten(), bins=60, alpha=0.5, density=True, label='After ReLU (expanded)', color='#E8A84C')
ax2.hist(x_final.flatten(), bins=60, alpha=0.5, density=True, label='Final (with residual)', color='#9B6DC8')
ax2.set_title('Activation Distributions\nThrough Transition Layer', fontsize=13)
ax2.set_xlabel('Activation value', fontsize=11)
ax2.set_ylabel('Density', fontsize=11)
ax2.legend(fontsize=11)
ax2.set_xlim(-4, 4)

print(f'Transition layer dimensions:')
print(f'  Input:    {c_demo}')
print(f'  Expanded: {c_exp_demo} (4x expansion)')
print(f'  Output:   {c_demo}')
print(f'\nSparsity after ReLU: {(h_expanded == 0).mean():.1%} of expanded activations are zero')

plt.tight_layout()
plt.show()

The left panel illustrates the bottleneck architecture: the representation is projected up to $4\times$ the channel dimension, passed through ReLU, then projected back down. The right panel shows the activation distributions at each stage.

The 4x expansion factor is important: it gives the network a much higher-dimensional space in which to compute nonlinear transformations before compressing back. The ReLU activation introduces sparsity (roughly half the expanded dimensions are zeroed out), which acts as an implicit feature selection mechanism. The residual connection ensures that the original information is preserved even if the transition layer makes only a small update.

---
## 8. Information Flow Through the Full Evoformer Stack

In AlphaFold2, the Evoformer consists of **48 stacked blocks**, each applying the full sequence of operations described above. Through this iterative refinement:

- **Early blocks** (1--12) tend to capture local patterns: sequence conservation, secondary structure signals, and nearby contacts.
- **Middle blocks** (13--36) propagate information over longer ranges, building up medium-range contacts and beginning to resolve the overall topology.
- **Late blocks** (37--48) refine the global fold, resolving long-range contacts and ensuring global consistency.

The triangular multiplicative updates are particularly important for this long-range propagation. In each block, contact information can propagate one step through an intermediate node. Over 48 blocks, this means a contact can propagate through up to 48 intermediaries, effectively allowing any pair of residues to influence each other regardless of sequence distance.

Let us simulate this iterative refinement with a simplified model.

In [ ]:
# ---------------------------------------------------------------
# Simulating Iterative Pair Representation Refinement Over 48 Blocks
# ---------------------------------------------------------------
np.random.seed(7)
N_res_sim = 40

# Create a ground-truth contact map for a simple protein fold
# (alpha-helix + beta-sheet topology)
true_contacts = np.zeros((N_res_sim, N_res_sim))

# Local contacts (backbone)
for i in range(N_res_sim - 1):
    true_contacts[i, i+1] = 1.0
    true_contacts[i+1, i] = 1.0

# Alpha-helix region (residues 0-15): i, i+4 contacts
for i in range(0, 12):
    true_contacts[i, i+4] = 1.0
    true_contacts[i+4, i] = 1.0

# Beta-sheet: antiparallel (residues 18-25 pair with 32-39)
for offset in range(8):
    i, j = 18 + offset, 39 - offset
    true_contacts[i, j] = 1.0
    true_contacts[j, i] = 1.0

# Initial pair representation: sparse, noisy version of contacts
# Only a few "seed" contacts are visible initially
pair_sim = np.random.rand(N_res_sim, N_res_sim) * 0.05
pair_sim = (pair_sim + pair_sim.T) / 2  # symmetrize

# Seed: only backbone and a few long-range contacts
for i in range(N_res_sim - 1):
    pair_sim[i, i+1] = 0.6
    pair_sim[i+1, i] = 0.6
# A few helix contacts as seeds
for i in [0, 4, 8]:
    pair_sim[i, i+4] = 0.4
    pair_sim[i+4, i] = 0.4
# One beta-sheet seed
pair_sim[20, 37] = 0.4
pair_sim[37, 20] = 0.4

def simplified_evoformer_step(pair, true_contacts, noise_scale=0.02):
    """Simplified Evoformer-like update that propagates contacts transitively."""
    N = pair.shape[0]
    
    # 1. Triangular propagation (simplified): z_ij += max_k min(z_ik, z_kj)
    update = np.zeros_like(pair)
    for i in range(N):
        for j in range(i+1, N):
            # Max over intermediaries of min(z_ik, z_kj)
            indirect = np.minimum(pair[i, :], pair[:, j])
            best_indirect = np.max(indirect)
            update[i, j] = best_indirect * 0.15
            update[j, i] = update[i, j]
    
    # 2. Small pull toward true contacts (simulating MSA-to-pair signal via OPM)
    opm_signal = true_contacts * 0.03
    
    # 3. Noise (simulating imperfect attention)
    noise = np.random.randn(N, N) * noise_scale
    noise = (noise + noise.T) / 2
    
    # Update with damping
    pair_new = pair * 0.9 + update + opm_signal + noise
    pair_new = np.clip(pair_new, 0, 1)
    pair_new = (pair_new + pair_new.T) / 2  # maintain symmetry
    np.fill_diagonal(pair_new, 1.0)
    
    return pair_new

# Run 48 iterations, saving snapshots
snapshots = {0: pair_sim.copy()}
pair_current = pair_sim.copy()
snapshot_iters = [1, 12, 24, 48]

for iteration in range(1, 49):
    pair_current = simplified_evoformer_step(pair_current, true_contacts,
                                              noise_scale=0.01 * (1 - iteration/48))
    if iteration in snapshot_iters:
        snapshots[iteration] = pair_current.copy()

# --- Visualization ---
fig, axes = plt.subplots(1, 5, figsize=(20, 4.5))

titles = [
    'Initial (block 0)\nSparse seeds',
    'Block 1\nLocal propagation',
    'Block 12\nMedium-range emerge',
    'Block 24\nLong-range contacts',
    'Block 48\nRefined contact map'
]

plot_iters = [0] + snapshot_iters

for idx, (it, title) in enumerate(zip(plot_iters, titles)):
    im = axes[idx].imshow(snapshots[it], cmap='hot_r', vmin=0, vmax=1)
    axes[idx].set_title(title, fontsize=11)
    axes[idx].set_xlabel('Residue $j$', fontsize=11)
    if idx == 0:
        axes[idx].set_ylabel('Residue $i$', fontsize=11)
    axes[idx].tick_params(labelsize=9)

plt.colorbar(im, ax=axes[-1], shrink=0.8, label='Contact strength')

fig.suptitle('Pair Representation Refinement Through 48 Evoformer Blocks\n'
             '(simulated: helix residues 0-15, antiparallel sheet residues 18-25/32-39)',
             fontsize=13, y=1.08)
plt.tight_layout()
plt.show()

# Quantitative: correlation with true contacts over iterations
print('\nCorrelation of pair representation with true contact map:')
for it in plot_iters:
    mask = np.triu(np.ones((N_res_sim, N_res_sim), dtype=bool), k=2)
    corr = np.corrcoef(snapshots[it][mask].flatten(), true_contacts[mask].flatten())[0, 1]
    print(f'  Block {it:>2d}: r = {corr:.3f}')

In [ ]:
# ---------------------------------------------------------------
# Convergence curve: correlation with true contacts over all 48 blocks
# ---------------------------------------------------------------
np.random.seed(7)
pair_track = pair_sim.copy()
correlations = []
mask_upper = np.triu(np.ones((N_res_sim, N_res_sim), dtype=bool), k=2)

correlations.append(np.corrcoef(pair_track[mask_upper], true_contacts[mask_upper])[0, 1])

for iteration in range(1, 49):
    pair_track = simplified_evoformer_step(pair_track, true_contacts,
                                           noise_scale=0.01 * (1 - iteration/48))
    corr = np.corrcoef(pair_track[mask_upper], true_contacts[mask_upper])[0, 1]
    correlations.append(corr)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(49), correlations, 'o-', color='#4C9BE8', markersize=4, linewidth=1.5)
ax.set_xlabel('Evoformer Block', fontsize=12)
ax.set_ylabel('Correlation with True Contact Map', fontsize=12)
ax.set_title('Convergence of Pair Representation\nOver 48 Evoformer Blocks', fontsize=13)
ax.set_xlim(-1, 49)
ax.set_ylim(0, 1.0)
ax.tick_params(labelsize=11)

# Annotate phases
ax.axvspan(0, 12, alpha=0.08, color='blue', label='Phase 1: Local patterns')
ax.axvspan(12, 36, alpha=0.08, color='orange', label='Phase 2: Medium-range')
ax.axvspan(36, 48, alpha=0.08, color='green', label='Phase 3: Refinement')
ax.legend(fontsize=11, loc='lower right')

plt.tight_layout()
plt.show()

print(f'Final correlation: {correlations[-1]:.3f}')

The simulation demonstrates three phases of refinement:

1. **Phase 1 (blocks 1--12):** Rapid improvement as local patterns propagate. The helix contacts fill in quickly because they are close in sequence and require few propagation steps.

2. **Phase 2 (blocks 12--36):** Steady improvement as medium-range and long-range contacts emerge. The beta-sheet contacts between residues 18--25 and 32--39 require multiple propagation steps to fully resolve.

3. **Phase 3 (blocks 36--48):** Refinement and convergence. The contact map is largely correct and the remaining iterations clean up noise and sharpen boundaries.

In the real Evoformer, this process is far more complex -- the attention mechanisms can propagate information over long ranges in a single step, the MSA representation provides a continuous stream of evolutionary signal via the outer product mean, and the learned weights are optimized end-to-end. Nevertheless, the qualitative behavior -- iterative refinement from sparse seeds to a complete contact map -- is a good approximation of what happens in practice.

---
## 9. Summary and Key Takeaways

The Evoformer is where AlphaFold2 does its "thinking." Through 48 iterations of specialized neural network operations, it transforms raw evolutionary and sequence information into rich representations that encode the protein's three-dimensional structure.

### Key operations and their roles

| Operation | Input | Output | Role |
|-----------|-------|--------|------|
| MSA Row Attention + Pair Bias | $\mathbf{m}, \mathbf{z}$ | $\mathbf{m}$ | Structure-aware sequence reasoning |
| MSA Column Attention | $\mathbf{m}$ | $\mathbf{m}$ | Cross-sequence communication (coevolution) |
| Outer Product Mean | $\mathbf{m}$ | $\Delta\mathbf{z}$ | MSA covariance $\to$ pair signal |
| Triangular Mult. Update (out/in) | $\mathbf{z}$ | $\mathbf{z}$ | Triangle consistency via gated multiplication |
| Triangular Self-Attention (start/end) | $\mathbf{z}$ | $\mathbf{z}$ | Triangle consistency via attention |
| Transition | $\mathbf{m}$ or $\mathbf{z}$ | same | Nonlinear feature transformation |

### Design principles

1. **Bidirectional communication:** The MSA and pair representations continuously exchange information. The pair bias injects structural knowledge into sequence reasoning; the outer product mean extracts coevolutionary signal into the pair representation.

2. **Triangle inequality as inductive bias:** The triangular updates (both multiplicative and attention-based) enforce that pairwise relationships are globally consistent. If $i$ is close to $k$ and $k$ is close to $j$, then $i$ should be close to $j$.

3. **Iterative refinement:** 48 blocks allow progressive refinement from local to global patterns. Early blocks capture obvious signals; later blocks resolve subtle, long-range interactions.

4. **Gating everywhere:** Every major operation uses sigmoid gating to control information flow. This allows the network to selectively update only the components that need updating at each step.

### What comes next

After 48 blocks, the Evoformer outputs:
- A refined pair representation $\mathbf{z} \in \mathbb{R}^{N_r \times N_r \times 128}$ encoding pairwise spatial relationships.
- A single-sequence representation $\mathbf{s} \in \mathbb{R}^{N_r \times 384}$ extracted from the first row of the MSA representation.

These are passed to the **Structure Module** (Notebook 6), which converts these abstract representations into explicit 3D coordinates -- the predicted protein structure.